# Galecopperbrug (MCDA)

This notebook presents a worked example of a Multi-Criteria Decision Analysis (MCDA) using **Preference Function Modeling (PFM)**. It evaluates alternative solutions to the Gold Coast beach erosion problem and demonstrates how stakeholders can assess design alternatives based on multiple criteria. These individual assessments are subsequently aggregated into an overall preference score for each alternative.

Many decision-making methods used in practice have a flawed mathematical foundation. These shortcomings often arise from the improper treatment of measurement scales, such as directly combining quantities expressed in different units (e.g., euros, kilograms, and square meters), aggregating ordinal data (e.g., 1st, 2nd, 3rd) with cardinal data (e.g., €1 million, €20 million, or 50 cm), or applying mathematical operations to scales for which they are not defined. For example, ranking aesthetics on a scale from 1 (best) to 4 (worst) does not imply that an alternative ranked 2 is twice as aesthetic as one ranked 4. Such practices implicitly assume measurement properties that the underlying data do not possess and can therefore lead to mathematically inconsistent results.

These issues are common in Multi-Criteria Decision Analysis (MCDA), where criteria often consist of a mixture of quantitative measures (e.g., costs or environmental impacts) and qualitative assessments (e.g., aesthetics or social acceptance). Arithmetic operations such as addition, averaging, and weighted summation are only meaningful when the underlying scales satisfy the required measurement properties. Applying these operations to ordinal scales can lead to mathematically inconsistent results, as different but equally valid numerical representations of the same preference ordering may produce different outcomes.

Preference Function Modeling addresses these challenges by first transforming each criterion onto a common preference scale prior to aggregation. This ensures that subsequent mathematical operations are meaningful, invariant to admissible transformations of the original data, and theoretically sound (Barzilai, 2010). This is achieved by eliciting preferences for each criterion and defining fixed reference points on the preference scale so that differences between alternatives become meaningful. A common approach, and the one used in this example, is to assign a value of 0 to the least preferred outcome and 100 to the most preferred outcome. The specific numerical values are not inherently important and may be changed (e.g., scales ranging from 50 to 150 are also used in the literature). What is essential is that the best and worst outcomes are explicitly defined and that intermediate alternatives are positioned proportionally between these fixed points.

Once stakeholder ratings have been correctly defined for each alternative, they must be aggregated into a single group preference score to support decision-making. PFM performs this aggregation using **affine aggregation**, which preserves the mathematical properties of the preference scales throughout the aggregation process.

This notebook demonstrates how to:

- Load and validate stakeholder ratings and criterion weights.
- Aggregate criterion ratings into a single preference score for each stakeholder (Level 1).
- Aggregate stakeholder preference scores into a single group preference score for each alternative (Level 2).

## Usage Guide

To perform an MCDA using PFM for your own System of Interest (SoI), you only need to modify the input **.csv** file containing the alternatives, criteria, stakeholder ratings, and criterion weights relevant to your case. A **.csv** (comma-separated values) file is a simple spreadsheet format that can be edited using software such as Microsoft Excel, LibreOffice Calc, or Visual Studio Code.

Provided that the structure of the input file is maintained, this notebook will automatically compute the aggregated preference scores for all stakeholders and the resulting group preference scores. Consequently, little to no modification of the notebook itself is required.

In [55]:
# Import packages
import numpy as np
import pandas as pd

# Round the float values in the dataframe to 2 decimal places
pd.options.display.float_format = '{:.2f}'.format

# Import local module for a-fine-aggregator
from genetic_algorithm_pfm.a_fine_aggregator import a_fine_aggregator

In [56]:
ratings = pd.read_csv(
    "MCDA_Ratings_Galecopperbrug.csv",
    sep=";",       # fields are semicolon-separated, not comma-separated
    skiprows=1     # skip the "MCDA Gold Coast Design Alternatives" title rowz
)

alternatives = list(ratings.columns[2:-1].unique()) # Get the list of alternatives from the dataframe columns, excluding the first two and last column
stakeholders = list(ratings["Stakeholder"].unique()) # Get the list of stakeholders from the "Stakeholder" column in the dataframe
print(f"Alternatives: {alternatives}")
print(f"Stakeholders: {stakeholders}")
display(ratings)


Alternatives: ['Complete Replacement', 'Convetional In-Situ Replacement', 'External Auxiliary Cables with Tower Extensions', 'Additional bridge']
Stakeholders: ['Rijkswaterstaat', 'Road Users', 'Local Residents', 'Environmental Organizations', 'Province', 'Maritime Sector', 'Contractors']


,Stakeholder,Criteria,Complete Replacement,Convetional In-Situ Replacement,External Auxiliary Cables with Tower Extensions,Additional bridge,Criteria_Weight
0,Rijkswaterstaat,Costs,10,100,80,0,0.25
1,Rijkswaterstaat,Accesibility during renovation,0,50,80,100,0.25
2,Rijkswaterstaat,Accesibility after renovation,80,80,80,100,0.25
3,Rijkswaterstaat,Sustainability,50,80,100,0,0.25
4,Road Users,Safety,100,100,0,100,0.25
5,Road Users,Longer Travel Time,0,0,100,100,0.75
6,Local Residents,Noise Polution,50,100,50,0,0.40
7,Local Residents,Felt Hinderance,0,20,100,0,0.60
8,Environmental Organizations,Enviromental impact,40,90,100,0,0.67
9,Environmental Organizations,Nature reservation,20,100,80,0,0.33


In [57]:
# Check each stakeholder's weights sum to 1 (i.e. 100%)
print("Check weights per stakeholder:")
all_valid = True

for stakeholder in stakeholders:
    stakeholder_weights = ratings.loc[ratings["Stakeholder"] == stakeholder, "Criteria_Weight"]
    total = stakeholder_weights.sum()
    is_valid = np.isclose(total, 1)
    all_valid &= is_valid

    status = "OK" if is_valid else "MISMATCH"
    print(f"  {stakeholder:<25s}: {total:6.2f}  [{status}]")


Check weights per stakeholder:
  Rijkswaterstaat          :   1.00  [OK]
  Road Users               :   1.00  [OK]
  Local Residents          :   1.00  [OK]
  Environmental Organizations:   1.00  [OK]
  Province                 :   1.00  [OK]
  Maritime Sector          :   1.00  [OK]
  Contractors              :   1.00  [OK]


In [58]:
# Set stakeholder weights
#               city, local, res, surf, tour
weights_dom =    [0.3, 0.1, 0.1, 0.1, 0.05, 0.2, 0.15]  # Designed varied weights for stakeholders
weights_eq =   [1/7, 1/7, 1/7, 1/7, 1/7, 1/7, 1/7]  # if equal weights for stakeholders (SUM of 1)

stakeholder_weights = weights_dom 

assert np.isclose(sum(stakeholder_weights), 1), f"Weights must sum to 1, got {sum(stakeholder_weights)}"


In [59]:
# Calculate the aggregated scores for each alternative using the a-fine-aggregator
# --- Level 1: aggregate criteria ratings -> one score per stakeholder per alternative ---
stakeholder_scores = {}

for stakeholder in stakeholders:
    stakeholder_data = ratings.loc[ratings["Stakeholder"] == stakeholder]
    criteria_weights = stakeholder_data["Criteria_Weight"].to_numpy()
    p = stakeholder_data[alternatives].to_numpy()  # shape: n_criteria x n_alternatives
    stakeholder_scores[stakeholder] = a_fine_aggregator(criteria_weights, p, scores_range=(-0.0, -100.0))

# Collect into matrix: rows = stakeholders, columns = alternatives (order matches `alternatives`)
stakeholder_score_matrix = np.array([stakeholder_scores[s] for s in stakeholders])

print("Individual stakeholder aggregated scores:")
display(pd.DataFrame(stakeholder_score_matrix, index=stakeholders, columns=alternatives))

# --- Level 2: aggregate stakeholder scores -> final preference score per alternative ---
final_scores = a_fine_aggregator(stakeholder_weights, stakeholder_score_matrix, scores_range=(-0.0, -100.0))

results = (
    pd.DataFrame(final_scores, index=alternatives, columns=["Preference score"])
    .round(2)
    .sort_values("Preference score", ascending=False)
)

print("Final aggregated preference scores per alternative:")
display(results)


Individual stakeholder aggregated scores:


,Complete Replacement,Convetional In-Situ Replacement,External Auxiliary Cables with Tower Extensions,Additional bridge
Rijkswaterstaat,0.00,82.95,100.00,67.16
Road Users,0.00,0.00,61.51,100.00
Local Residents,27.99,70.39,100.00,0.00
Environmental Organizations,35.83,99.72,100.00,0.00
Province,0.00,50.00,100.00,100.00
Maritime Sector,81.04,81.04,100.00,0.00
Contractors,78.90,0.00,69.57,100.00


Final aggregated preference scores per alternative:


,Preference score
External Auxiliary Cables with Tower Extensions,100.00
Convetional In-Situ Replacement,45.42
Additional bridge,27.15
Complete Replacement,0.00


## Interpreting the Results

The `results` table ranks the four alternatives by their final group preference score, from most to least preferred across all five stakeholder groups combined. Because the underlying scores are normalized before being combined, the results reflect each alternative's *relative* standing among the options considered — not an absolute measure of quality.

It's worth revisiting the `stakeholder_weights` (and, further upstream, each stakeholder's `Criteria_Weight` values in the CSV) to see how sensitive the final ranking is to these assumptions — a common next step in an MCDA is a simple sensitivity or scenario analysis.
